# Practical P13: Rate Limit Retry Logic
**Learning Outcome**: Build production-grade retry logic for any API.

## Part 1: Exponential Backoff Pattern
When hitting rate limits (`HTTP 429`), standard code will crash. We handle this by retrying the request with increasing delays between attempts.
Formula: `delay = initial_wait * (2 ** attempt)`.


In [ ]:
import time
import random

def simulate_rate_limit_api(attempt_history=[]):
    # Succeeds only on 3rd attempt
    attempt_history.append(1)
    if len(attempt_history) < 3:
        raise ValueError('HTTP 429 Too Many Requests')
    return 'Success output!'


## Part 2: Implementing a Retry Decorator
Decorators allow us to apply retry logic across multiple query functions cleanly.


In [ ]:
def retry_on_failure(max_retries=5, initial_wait=1):
    def decorator(func):
        def wrapper(*args, **kwargs):
            wait = initial_wait
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as err:
                    if attempt == max_retries - 1:
                        print('Max retries exceeded!')
                        raise
                    # Jitter: add random millisecond delays to prevent thundering herd issues
                    jitter = random.uniform(0, 0.1)
                    actual_wait = wait + jitter
                    print(f"Warning: {err}. Retrying in {actual_wait:.2f} seconds...")
                    time.sleep(actual_wait)
                    wait *= 2
        return wrapper
    return decorator


## Hands-On Exercise
**Task**: Apply the `@retry_on_failure` decorator to a function that calls `simulate_rate_limit_api`. Verify that the script retries on failures and finishes successfully on the 3rd attempt.


In [ ]:
# TODO: Decorate and test function
history = []

@retry_on_failure(max_retries=4, initial_wait=1)
def execute_query():
    return simulate_rate_limit_api(history)

res = execute_query()
print('Final Output:', res)
